<a href="https://colab.research.google.com/github/imets01/synthetic_network_data_gen/blob/main/low_level_features/WGAN/WGAN_postprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to post-process the output of WGAN to get a realistic synthetic data output

### 1. Loading the trained model

We have to have the artifacts of the model we want to load ready:
- condition_scaler.gz
- sequence_scaler.gz
- critic.pth
- generator.pth
- sequence_columns.json

If we are running this notebook in Colab - upload all these in a folder named WGAN


First we clone the github repo to have all the data here.

In [9]:
!git clone https://github.com/imets01/synthetic_network_data_gen.git

Cloning into 'synthetic_network_data_gen'...
remote: Enumerating objects: 1017, done.
remote: Counting objects: 100% (296/296), done.
remote: Compressing objects: 100% (217/217), done.
remote: Total 1017 (delta 87), reused 256 (delta 64), pack-reused 721 (from 4)
Receiving objects: 100% (1017/1017), 560.86 MiB | 36.16 MiB/s, done.
Resolving deltas: 100% (460/460), done.
Updating files: 100% (41/41), done.


In [10]:
%cd synthetic_network_data_gen
!git fetch
!git checkout anna

/content/synthetic_network_data_gen
Branch 'anna' set up to track remote branch 'anna' from 'origin'.
Switched to a new branch 'anna'


Then we need to have the same dataset setup that we used during the training. For this we are installing the necessary libraries and import them

In [ ]:
%pip in=============

ERROR: unknown command "in============="


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import zipfile
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import torch.autograd as autograd
import os
import joblib
import json

In [2]:
class QuicSequenceDataset(Dataset):
    def __init__(self, high_level_csv, low_level_zip_path, folder_name_in_zip='separate_low_level_files'):

        self.high_level_df = pd.read_csv(high_level_csv)
        self.high_level_df = self.high_level_df.replace([np.inf, -np.inf], np.nan).dropna(how='any')

        # Use a copy of flow_ids as a string Series for reliable mapping/filtering
        self.flow_ids_int = self.high_level_df['file_id'].copy()

        self.low_level_zip_path = low_level_zip_path
        self.folder_name_in_zip = folder_name_in_zip

        to_keep = [
            'implementation', 'connection_duration',  'version_negotiation_occurred', 'retry_occurred', 'migration_type',
            'first_path_validation_response_latency', 'path_validation_initiated', 'packets_sent_client',
            'packets_sent_server', 'handshake_duration', 'time_to_migration', 'migration_duration', # Corrected
            'packets_before_migration', 'total_bidi_streams_client_init',
            'total_udi_streams_client_init',
            # 'connection_close_type'
        ]
        self.condition_features_df = self.high_level_df.copy() # Work on a copy

        for col in to_keep:
            if col not in self.condition_features_df.columns:
                 self.condition_features_df[col] = 0

        self.condition_features_df = self.condition_features_df[to_keep + ['file_id']]

        categorical_cols = ['migration_type', 'implementation']
        existing_categorical_cols = [col for col in categorical_cols if col in self.condition_features_df.columns]
        for col in existing_categorical_cols:
            if self.condition_features_df[col].dtype == 'object':
                self.condition_features_df[col] = self.condition_features_df[col].astype('category')

        categorical_cols_to_dummy = self.condition_features_df.select_dtypes(include=['category']).columns
        self.condition_features_df = pd.get_dummies(self.condition_features_df, columns=categorical_cols_to_dummy, prefix=categorical_cols_to_dummy)


        self.condition_scaler = MinMaxScaler(feature_range=(-1, 1))
        self.sequence_scaler = MinMaxScaler(feature_range=(-1, 1))

        # New list to hold ALL sequence TENSORS in memory
        self.preloaded_sequences = []
        self.flow_id_to_path = {}
        self.sequence_columns = None

        # The list of flow IDs we need to process
        target_flow_ids = set(self.flow_ids_int.astype(str))

        # 1. Create temporary lists to hold validated data
        temp_sequences = []
        temp_valid_ids = []
        dropped_count = 0

        with zipfile.ZipFile(self.low_level_zip_path, 'r') as zf:
            zip_names = zf.namelist()
            target_prefix = f"{self.folder_name_in_zip}/"

            # Map potential files first
            potential_files = {}
            for name in zip_names:
                if name.startswith(target_prefix) and name.endswith('.csv'):
                    relative_name = name[len(target_prefix):]
                    flow_id_match = relative_name.split('_')[0]
                    if flow_id_match in target_flow_ids:
                        potential_files[int(flow_id_match)] = name
                        target_flow_ids.discard(flow_id_match)

            potential_ids = list(potential_files.keys())

            # Loop through potential IDs, load data, CHECK VALIDITY, then store
            for flow_id in tqdm(potential_ids, desc="Loading & Validating Sequences"):
                file_name_in_zip = potential_files[flow_id]

                with zf.open(file_name_in_zip) as f:
                    try:
                        df = pd.read_csv(f).drop('frame_number', axis=1)
                        vals = df.values

                        # --- NEGATIVE VALUE CHECK ---
                        if (vals < 0).any():
                            dropped_count += 1
                            continue # Skip this sequence and do not add to valid lists
                        # ----------------------------

                        if self.sequence_columns is None:
                            self.sequence_columns = df.columns.tolist()

                        # Only add if check passed
                        temp_sequences.append(vals)
                        temp_valid_ids.append(flow_id)
                        self.flow_id_to_path[flow_id] = file_name_in_zip

                    except Exception as e:
                        print(f"Error processing {file_name_in_zip}: {e}")
                        continue

        print(f"Dropped {dropped_count} sequences containing negative values.")

        # 2. Update the High-Level Dataframe to match valid sequences
        # This removes the rows (conditions) corresponding to the dropped negative sequences
        self.flow_ids = pd.Series(temp_valid_ids)
        self.condition_features_df = self.condition_features_df[self.condition_features_df['file_id'].isin(temp_valid_ids)].reset_index(drop=True)

        # 3. Fit scaler on ALL data
        if temp_sequences:
            full_sequence_data = np.concatenate(temp_sequences, axis=0)

            if np.isnan(full_sequence_data).any() or np.isinf(full_sequence_data).any():
                print("FATAL ERROR: NaN or INF found in the unscaled sequence data.")

            print(f"Unscaled Sequence Data Min/Max: {full_sequence_data.min():.4f} / {full_sequence_data.max():.4f}")
            self.sequence_scaler.fit(full_sequence_data)

            # Fit condition scaler on the filtered condition dataframe
            condition_features_for_scaling = self.condition_features_df.drop('file_id', axis=1)
            self.scaled_conditions = self.condition_scaler.fit_transform(condition_features_for_scaling)
            self.scaled_conditions = torch.FloatTensor(self.scaled_conditions)

            # Transform sequences to tensors
            for unscaled_seq in tqdm(temp_sequences, desc="Scaling Sequences to Tensors"):
                scaled_seq = self.sequence_scaler.transform(unscaled_seq)
                if scaled_seq.min() < -1.01 or scaled_seq.max() > 1.01:
                    print("WARNING: SCALED DATA OUT OF RANGE!")
                self.preloaded_sequences.append(torch.FloatTensor(scaled_seq))
        else:
            print("Error: No valid sequence data found!")
            return

        # if self.condition_features_df.isnull().values.any() or np.isinf(self.condition_features_df.values).any():
        #     print("FATAL ERROR: NaN or INF found in the condition data.")

    def __len__(self):
        return len(self.preloaded_sequences)

    def __getitem__(self, idx):
        scaled_sequence = self.preloaded_sequences[idx]
        scaled_condition = self.scaled_conditions[idx]

        return {
            'sequence': scaled_sequence,
            'condition': scaled_condition
        }

In [3]:
class Generator(nn.Module):
    def __init__(self, latent_dim, condition_dim, sequence_feature_dim, hidden_dim=64, num_layers=1):
        super().__init__()
        self.rnn = nn.LSTM(input_size=condition_dim, hidden_size=hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc_init_h = nn.Linear(latent_dim + condition_dim, hidden_dim * num_layers)
        self.fc_init_c = nn.Linear(latent_dim + condition_dim, hidden_dim * num_layers)
        self.fc_out = nn.Linear(hidden_dim, sequence_feature_dim)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, noise, condition, seq_len):
        batch_size = noise.shape[0]
        combined_input = torch.cat([noise, condition], dim=1)
        h0 = self.fc_init_h(combined_input).view(self.num_layers, batch_size, self.hidden_dim)
        c0 = self.fc_init_c(combined_input).view(self.num_layers, batch_size, self.hidden_dim)
        initial_state = (h0, c0)
        condition_expanded = condition.unsqueeze(1).repeat(1, seq_len, 1)
        rnn_out, _ = self.rnn(condition_expanded, initial_state)

        output = torch.tanh(self.fc_out(rnn_out))
        return output

We also need to define where the training data was, in case we would like to try the model with the training data samples.

In [4]:
high_level_csv = '/content/synthetic_network_data_gen/high_level_features/dataset/all_captures_dataset.csv'
low_level_dir = '/content/synthetic_network_data_gen/low_level_features/dataset/separate_low_level_files.zip'

In [5]:
def post_process_generated_data(generated_sequences_tensor,
                                high_level_conditions_tensor,
                                sequence_scaler,
                                condition_scaler,
                                sequence_columns,
                                condition_columns_after_dummies):

    # --- Step 1: Inverse Transform Data ---
    generated_sequences_np = generated_sequences_tensor.squeeze(0).cpu().detach().numpy()
    high_level_conditions_np = high_level_conditions_tensor.cpu().detach().numpy()

    # Inverse scale back to the original [-1, 1] range to real values
    unscaled_sequences = sequence_scaler.inverse_transform(generated_sequences_np)
    unscaled_conditions = condition_scaler.inverse_transform(high_level_conditions_np)

    low_level_df = pd.DataFrame(unscaled_sequences, columns=sequence_columns)

    high_level_df = pd.DataFrame(unscaled_conditions, columns=condition_columns_after_dummies)

    # --- Step 2: Handle Categorical and Binary Features ---
    for col in [c for c in low_level_df.columns if c!= 'delta_time']:
        low_level_df[col] = low_level_df[col].round().astype(int)

    # Handle one-hot encoded features using argmax
    # Example for 'migration_type' if it were in the low-level data (adapt as needed)
    # migration_cols = [c for c in low_level_df.columns if c.startswith('migration_type_')]
    # if migration_cols:
    #     low_level_df['migration_type'] = low_level_df[migration_cols].idxmax(axis=1)
    #     low_level_df = low_level_df.drop(columns=migration_cols)

    # --- Step 3: Enforce Deterministic Rules ---

    low_level_df.insert(0, 'frame_number', np.arange(1, len(low_level_df) + 1))
    handshake_duration = low_level_df[low_level_df['count_handshake']>0]['delta_time'].sum()
    print(f'handshake_duration: {handshake_duration}')

    total_duration = low_level_df['delta_time'].sum()
    print(f'total_duration: {total_duration}')

    time_to_migration = low_level_df[low_level_df['frame_number']<14]['delta_time'].sum()
    print(f'time_to_migration: {time_to_migration}')

    # (Optional but recommended) Reconstruct packet direction if using the "burst" method
    # if 'is_burst_start' in low_level_df.columns:
    #     ... (insert burst reconstruction logic here) ...

    # --- Step 4: Reconcile Summary Statistics ---
    # This is the crucial step for consistency. We update the high-level features
    # to match the reality of what was actually generated in the low-level sequence.

    # Example: Recalculate packet and byte counts
    # You would need to add 'packet_direction' and 'packet_length' to your low-level features
    if 'packet_direction' in low_level_df.columns and 'packet_length' in low_level_df.columns:
        packets_sent_client = len(low_level_df[low_level_df['packet_direction'] == 0])
        packets_sent_server = len(low_level_df[low_level_df['packet_direction'] == 1])
        bytes_sent_client = low_level_df.loc[low_level_df['packet_direction'] == 0, 'packet_length'].sum()
        bytes_sent_server = low_level_df.loc[low_level_df['packet_direction'] == 1, 'packet_length'].sum()

        # Update the high-level DataFrame with the new, consistent values
        # Note: The column names here must exactly match those from your condition_scaler
        if 'quic_packets_sent_client' in high_level_df.columns:
            print(f'quic_packets_sent_client originally: {high_level_df['quic_packets_sent_client']}')
            high_level_df['quic_packets_sent_client'] = packets_sent_client
            print(f'quic_packets_sent_client after change: {high_level_df['quic_packets_sent_client']}')
        if 'quic_packets_sent_server' in high_level_df.columns:
            print(f'packets_sent_server originally: {high_level_df['packets_sent_server']}')
            high_level_df['quic_packets_sent_server'] = packets_sent_server
            print(f'packets_sent_server after change: {high_level_df['packets_sent_server']}')

        # if 'bytes_sent_client' in high_level_df.columns:
        #      high_level_df['bytes_sent_client'] = bytes_sent_client
        # if 'bytes_sent_server' in high_level_df.columns:
        #      high_level_df['bytes_sent_server'] = bytes_sent_server

    return low_level_df, high_level_df

In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
dataset.condition_features_df.head()

,connection_duration,version_negotiation_occurred,retry_occurred,first_path_validation_response_latency,path_validation_initiated,packets_sent_client,packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,total_bidi_streams_client_init,total_udi_streams_client_init,file_id,implementation_aioquic,implementation_nginx,implementation_quicgo,implementation_quiche,migration_type_AFTER_DOWNLOAD,migration_type_BEFORE_DOWNLOAD,migration_type_DURING_DOWNLOAD,migration_type_NO_DOWNLOAD,migration_type_NO_MIGRATION
0,49.246073,0,0,14.654160,1.0,7,7,16.601086,28.691053,15.407085,7,1,4,3172,True,False,False,False,False,True,False,False,False
1,22.886992,0,0,9.438992,1.0,8,6,9.938002,9.589911,10.220051,5,1,4,3173,True,False,False,False,False,True,False,False,False
2,24.952888,0,0,10.677099,1.0,8,6,10.686874,9.984970,11.665106,5,1,4,3174,True,False,False,False,False,True,False,False,False
3,31.651020,0,0,14.860868,1.0,6,5,11.970997,12.630939,15.383959,6,1,4,3175,True,False,False,False,False,True,False,False,False
4,35.304070,0,0,14.575958,1.0,7,7,10.221004,17.075062,15.029907,7,1,4,3176,True,False,False,False,False,True,False,False,False


In [7]:
[49.246073, #'connection_duration',
0, #  'version_negotiation_occurred',
0,#  'retry_occurred',
14.654160 ,#  'first_path_validation_response_latency',
1.0, #  'path_validation_initiated',
7,#  'packets_sent_client',
7,#  'packets_sent_server',
16.6,#  'handshake_duration',
28.7,#  'time_to_migration',
15.4,#  'migration_duration',
7,#  'packets_before_migration',
1,#  'total_bidi_streams_client_init',
4,#  'total_udi_streams_client_init',
1,#  'implementation_aioquic',
0,#  'implementation_nginx',
0,#  'implementation_quicgo',
0,#  'implementation_quiche',
0,#  'migration_type_AFTER_DOWNLOAD',
1,#  'migration_type_BEFORE_DOWNLOAD',
0,#  'migration_type_DURING_DOWNLOAD',
0,#  'migration_type_NO_DOWNLOAD',
0#  'migration_type_NO_MIGRATION'
 ]

[49.246073,
 0,
 0,
 14.65416,
 1.0,
 7,
 7,
 16.6,
 28.7,
 15.4,
 7,
 1,
 4,
 1,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0]

In [ ]:
dataset.condition_features_df.columns.to_list()

['connection_duration',
 'version_negotiation_occurred',
 'retry_occurred',
 'first_path_validation_response_latency',
 'path_validation_initiated',
 'packets_sent_client',
 'packets_sent_server',
 'handshake_duration',
 'time_to_migration',
 'migration_duration',
 'packets_before_migration',
 'total_bidi_streams_client_init',
 'total_udi_streams_client_init',
 'file_id',
 'implementation_aioquic',
 'implementation_nginx',
 'implementation_quicgo',
 'implementation_quiche',
 'migration_type_AFTER_DOWNLOAD',
 'migration_type_BEFORE_DOWNLOAD',
 'migration_type_DURING_DOWNLOAD',
 'migration_type_NO_DOWNLOAD',
 'migration_type_NO_MIGRATION']

In [13]:
print("--- Loading pre-trained models and scalers for synthesis ---")

# --- Step 1: Define paths and parameters (MUST match training) ---
load_dir = "/content/WGAN"
generator_path = os.path.join(load_dir, "generator.pth")
sequence_scaler_path = os.path.join(load_dir, "sequence_scaler.gz")
condition_scaler_path = os.path.join(load_dir, "condition_scaler.gz")
sequence_columns_path = os.path.join(load_dir, "sequence_columns.json")

# Re-define model parameters
LATENT_DIM = 20
HIDDEN_DIM = 256
NUM_LAYERS_GEN = 2

# automatically select 'cuda' if available, otherwise it will fall back to 'cpu'.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



--- Loading pre-trained models and scalers for synthesis ---
Using device: cuda


In [14]:
# --- Step 2: Load the scalers and columns ---
sequence_scaler = joblib.load(sequence_scaler_path)
condition_scaler = joblib.load(condition_scaler_path)

with open(sequence_columns_path, 'r') as f:
    sequence_columns = json.load(f)

sequence_feature_dim = sequence_scaler.n_features_in_
condition_dim = condition_scaler.n_features_in_

# --- Step 3: Instantiate model and load saved weights with map_location ---
# First, create an instance of the model structure (this is always done on the CPU initially)
loaded_generator = Generator(LATENT_DIM, condition_dim, sequence_feature_dim, HIDDEN_DIM, num_layers=NUM_LAYERS_GEN)


loaded_generator.load_state_dict(torch.load(generator_path, map_location=device))

loaded_generator.to(device)

print("Generator loaded successfully.")



Generator loaded successfully.


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.0.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [16]:
dataset = QuicSequenceDataset(high_level_csv, low_level_dir)

Loading & Validating Sequences: 100%|██████████| 11897/11897 [00:24<00:00, 490.66it/s]


Dropped 125 sequences containing negative values.
Unscaled Sequence Data Min/Max: 0.0000 / 1398.0000


Scaling Sequences to Tensors: 100%|██████████| 11772/11772 [00:02<00:00, 5086.75it/s]


In [19]:
# --- Step 4: Run the synthesis process ---

print("\n--- Generating and Refining a New Sample ---")
loaded_generator.eval()
with torch.no_grad():
    data = [
        29.246073, #'connection_duration',
        1, #'version_negotiation_occurred',
        1,#'retry_occurred',
        14.654160 ,#first_path_validation_response_latency',
        1.0, #path_validation_initiated',
        7,#  'packets_sent_client',
        7,#  'packets_sent_server',
        16.6,#  'handshake_duration',
        15.7,#  'time_to_migration',
        3.4,#  'migration_duration',
        7,#  'packets_before_migration',
        1,#  'total_bidi_streams_client_init',
        4,#  'total_udi_streams_client_init',
        1,#  'implementation_aioquic',
        0,#  'implementation_nginx',
        0,#  'implementation_quicgo',
        0,#  'implementation_quiche',
        1,#  'migration_type_AFTER_DOWNLOAD',
        0,#  'migration_type_BEFORE_DOWNLOAD',
        0,#  'migration_type_DURING_DOWNLOAD',
        0,#  'migration_type_NO_DOWNLOAD',
        0#  'migration_type_NO_MIGRATION'
 ]

    tensor = np.array(data, dtype=float)

    tensor = tensor.reshape(1, -1)
    sample_condition_scaled = torch.FloatTensor(condition_scaler.transform(tensor)).to(device)

    # Get a sample condition and move it to the correct device
    #sample_condition_scaled = torch.FloatTensor(dataset.scaled_conditions[0]).unsqueeze(0).to(device)

    noise = torch.randn(1, LATENT_DIM, device=device)
    desired_seq_len = 26

    generated_sequence_scaled = loaded_generator(noise, sample_condition_scaled, desired_seq_len)

    # Use your post-processing function (it's designed to be device-agnostic)
    final_low_level_df, final_high_level_df = post_process_generated_data(
        generated_sequences_tensor=generated_sequence_scaled,
        high_level_conditions_tensor=sample_condition_scaled,
        sequence_scaler=sequence_scaler,
        condition_scaler=condition_scaler,
        sequence_columns=sequence_columns,
        condition_columns_after_dummies=condition_scaler.get_feature_names_out()
    )

    print("\n--- Final, Cleaned Synthetic Data ---")
    print("\nReconciled High-Level Features:")
    display(final_high_level_df)
    print("\nGenerated Low-Level Packet Sequence:")
    display(final_low_level_df)


--- Generating and Refining a New Sample ---
handshake_duration: 0.0003136955783702433
total_duration: 1.9713780879974365
time_to_migration: 0.001267526182346046

--- Final, Cleaned Synthetic Data ---

Reconciled High-Level Features:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,connection_duration,version_negotiation_occurred,retry_occurred,first_path_validation_response_latency,path_validation_initiated,packets_sent_client,packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,total_bidi_streams_client_init,total_udi_streams_client_init,implementation_aioquic,implementation_nginx,implementation_quicgo,implementation_quiche,migration_type_AFTER_DOWNLOAD,migration_type_BEFORE_DOWNLOAD,migration_type_DURING_DOWNLOAD,migration_type_NO_DOWNLOAD,migration_type_NO_MIGRATION
0,29.246096,1.0,1.0,14.65416,1.0,7.0,7.0,16.599997,15.700001,3.400001,7.0,1.0,4.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



Generated Low-Level Packet Sequence:


,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,count_vn,count_ack,count_padding,count_connection_close,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_ping,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,4.982243e-05,1092,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,2,1.874105e-04,127,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
2,3,3.201956e-07,1354,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,1.888513e-04,1348,1,1,1,0,1,0,0,0,1,1,0,0,0,0,0,0,4,0,0,0,0,0
4,5,3.922396e-05,278,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0
5,6,8.562029e-05,1380,0,1,1,0,1,1,0,0,2,1,0,0,0,2,0,0,1,0,0,0,0,0
6,7,6.746841e-04,590,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,2,1,2,0,8,1
7,8,1.665017e-05,124,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8,9,4.578797e-06,1372,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
9,10,4.866973e-06,1258,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0


In [ ]:
with torch.no_grad():
    # Load your original dataset just to get a sample condition
    # (In a real application, you might generate this condition synthetically)
    # high_level_csv = '...'
    # low_level_zip_path = '...'
    dataset = QuicSequenceDataset(high_level_csv, low_level_dir)
    data = [
      1.0,        # retry occurred
      1.0,        # version negotiation occurred
      16.60674,   # handshake_duration
      28.69251,  # time_to_migration
      15.4074,     # migration_duration
      7.0,       # packets_before_migration
      1.0,        # total_bidi_streams_client_init
      4.0,        # total_udi_streams_client_init
      1.0,        # path_validation_initiated
      1.0         # migration_type_IP_AND_PORT
    ]

    tensor = np.array(data, dtype=float)

    tensor = tensor.reshape(1, -1)
    sample_condition_scaled = torch.FloatTensor(condition_scaler.transform(tensor)).to(device)

    # Get a sample condition and move it to the correct device
    #sample_condition_scaled = torch.FloatTensor(dataset.scaled_conditions[0]).unsqueeze(0).to(device)

    noise = torch.randn(1, LATENT_DIM, device=device)
    desired_seq_len = 26

    generated_sequence_scaled = loaded_generator(noise, sample_condition_scaled, desired_seq_len)

    # Use your post-processing function (it's designed to be device-agnostic)
    final_low_level_df, final_high_level_df = post_process_generated_data(
        generated_sequences_tensor=generated_sequence_scaled,
        high_level_conditions_tensor=sample_condition_scaled,
        sequence_scaler=sequence_scaler,
        condition_scaler=condition_scaler,
        sequence_columns=sequence_columns,
        condition_columns_after_dummies=condition_scaler.get_feature_names_out()
    )

    print("\n--- Final, Cleaned Synthetic Data ---")
    print("\nReconciled High-Level Features:")
    display(final_high_level_df)
    print("\nGenerated Low-Level Packet Sequence:")
    display(final_low_level_df)

handshake_duration: 0.004330658353865147
total_duration: 0.015452152118086815
time_to_migration: 0.008498572744429111

--- Final, Cleaned Synthetic Data ---

Reconciled High-Level Features:


/tmp/ipython-input-705003398.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.condition_features_df['migration_type'] = self.condition_features_df['migration_type'].astype('category')
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


,retry_occurred,version_negotiation_occurred,handshake_duration,time_to_migration,migration_duration,packets_before_migration,total_bidi_streams_client_init,total_udi_streams_client_init,path_validation_initiated,migration_type_IP_AND_PORT
0,1.0,1.0,16.606741,28.692509,15.4074,7.0,1.0,4.0,1.0,1.0



Generated Low-Level Packet Sequence:


,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,count_vn,count_ack,count_padding,count_connection_close,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000420,617,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,0.000274,75,1,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0
2,3,0.001539,515,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,0.000408,97,1,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,5,0.000653,985,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
5,6,0.001900,1128,1,1,1,0,1,0,0,0,1,0,0,0,0,0,0,2,0,0,0,0,0
6,7,0.000063,523,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
7,8,0.002368,1375,0,1,1,0,1,1,0,0,2,1,0,0,0,1,0,1,0,0,0,0,0
8,9,0.000578,360,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,1,1,1,0,25,1
9,10,0.000109,75,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1


In [ ]:
final_low_level_df.columns